# Self-Representation Experiment — Full Pipeline

This notebook tests whether **LLaMA-3.1-8B has a dedicated internal representation of "self"**, a mechanistic interpretability question about whether the model encodes something like a self-concept in its weights.

The pipeline works by:
1. Generating prompts that ask the model to reason as different entity types (self, human, animal, object)
2. Recording the model's internal activations at every layer for each prompt
3. Finding a "self/other direction" in activation space that separates self-representation from others
4. Checking that this direction isn't just a surface-level confound (grammar, roleplay framing, animacy)
5. Visualising the results

This experiment runs with **A100 GPU**

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — make sure you selected a GPU runtime.')

## 1. Mount Google Drive
Save activations, directions, and figures into your Google Drive so you don't have to re-run if the session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ureca26_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Outputs will be saved to: {DRIVE_DIR}')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/nmat2010/ureca26.git'  
REPO_DIR = '/content/ureca26'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies.
# Ignore numpy/jax/opencv conflict warningsthat don't affect this experiment.
!pip install -e . -q

## 3. HuggingFace login

LLaMA-3.1-8B is a gated model so you need a HuggingFace account with access approved.

Store your token as a Colab Secret named `HF_TOKEN` (click the key icon in the left sidebar) or enter the token in the interactive prompt below

If you don't have access yet: request it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A: read from Colab Secrets (recommended)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Colab secret.')
except Exception:
    # Option B: interactive prompt — will ask you to paste your token
    login()

## 4. Configure output paths

The default config in `configs/experiment.yaml` writes outputs to local paths like `data/` and `figures/`. Here we redirect all outputs to Google Drive so they persist across sessions. A temporary config file is written to `/tmp/` for this session.

In [ ]:
import yaml, os

CONFIG_PATH = 'configs/experiment.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['paths']['data_dir']        = f'{DRIVE_DIR}/data'
cfg['paths']['activations_dir'] = f'{DRIVE_DIR}/data/activations'
cfg['paths']['directions_dir']  = f'{DRIVE_DIR}/data/directions'
cfg['paths']['figures_dir']     = f'{DRIVE_DIR}/figures'
cfg['paths']['prompts_file']    = f'{DRIVE_DIR}/data/prompts.json'

COLAB_CONFIG = '/tmp/experiment_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

for d in cfg['paths'].values():
    if not d.endswith('.json'):
        os.makedirs(d, exist_ok=True)

print('Config written to', COLAB_CONFIG)
print('Output dirs created under', DRIVE_DIR)

## 5. Step 01 — Generate prompts

`01_generate_prompts.py` builds the dataset of text prompts used to probe the model.

It generates scenarios across **8 domains** (problem-solving, decision-making, social reasoning, task planning, self-assessment, knowledge boundaries, authority deference, tool use) for **5 entity classes** (self, expert human, average human, animal, object). Each prompt frames the same scenario from the perspective of one of these entities — e.g. *"As an AI assistant, how would you approach this problem?"* vs *"As a dog, how would you approach this problem?"*

Outputs:
- `data/prompts.json` — all prompts with metadata (entity class, domain, train/test split)
- `data/config_snapshot.yaml` — a copy of the config used

In [ ]:
!python scripts/01_generate_prompts.py --config {COLAB_CONFIG}

## 6. Step 02 — Extract activations

`02_extract_activations.py` loads LLaMA-3.1-8B and runs every prompt through it, recording the **residual stream activations** at every layer (all 32 layers × 4096 hidden dimensions) for each prompt.

This is the core data collection step. Think of it as asking: *"what does the model's internal state look like at each layer when it's thinking about 'self' vs 'an animal'?"* We capture this at two token positions:
- **`final`** — the last token of the prompt (what the model is "thinking" right before generating a response)
- **`entity`** — the token corresponding to the entity word itself (e.g. "I", "dog", "expert")

Outputs (saved to `data/activations/`):
- `*_core.h5` — activations for the 200 main scenarios × 5 entity classes
- `*_ctrl_grammatical_person.h5` — control: first-person vs third-person phrasing
- `*_ctrl_role_play.h5` — control: roleplay framing
- `*_ctrl_animacy.h5` — control: living vs non-living entities

In [ ]:
!python scripts/02_extract_activations.py \
    --config {COLAB_CONFIG} \
    --batch_size 4


## 7. Step 03 — Find self/other directions

`03_find_directions.py` analyses the saved activations to find a **self/other direction** — a vector in the model's activation space that separates "thinking about self" from "thinking about other entities".

It does this two ways at each of the 32 layers:
- **Mean-difference direction** — the vector pointing from the average "other" activation to the average "self" activation
- **Logistic probe** — a linear classifier trained to predict entity class from activations, with cross-validated C hyperparameter tuning. The probe's weight vector is another candidate direction.

The best layer is chosen by probe test accuracy. It also computes pairwise cosine similarities between entity-class directions to see how geometrically distinct they are.

Outputs (saved to `data/directions/<model>/<token_position>/`):
- `direction_results.pkl` — directions, probe weights, accuracies, similarity matrices

In [ ]:
!python scripts/03_find_directions.py --config {COLAB_CONFIG}

## 8. Step 04 — Test specificity

`04_test_specificity.py` is the key validity check: **is the self/other direction actually about self-representation, or is it just picking up on surface-level features?**

It tests three confounds by measuring the cosine similarity between the self/other direction and each confound direction:
- **Grammatical person** — is the direction just detecting first-person ("I") vs third-person ("they") pronoun use?
- **Role-play** — is it just detecting roleplay framing in the prompt?
- **Animacy** — is it just detecting living vs non-living entities?

It also trains a **residual probe** — a classifier on activations with the self/other direction projected out — to check how much self-representation information remains after removing the direction. A large drop in accuracy = the direction captures most of the self-representation signal.

Bootstrap statistics (n=1000) are used to get confidence intervals on all similarity scores.

A good result: **low cosine similarity** with all three confounds, meaning the self/other direction is specific to self-representation rather than a surface artefact.

Outputs: `specificity_results.pkl`

In [ ]:
!python scripts/04_test_specificity.py --config {COLAB_CONFIG}

## 9. Step 05 — Visualise results

`05_visualize.py` loads the direction and specificity results and generates publication-quality figures, including:
- **Probe accuracy by layer** — which layers best linearly separate entity classes
- **Pairwise direction similarity heatmap** — how geometrically distinct the entity-class directions are from each other
- **Confound cosine similarity by layer** — how much the self/other direction overlaps with grammatical person, animacy, and roleplay directions at each layer
- **Residual probe accuracy** — how much self-representation information survives after projecting out the self/other direction

Figures are saved to `figures/` on Google Drive and displayed inline below.

In [ ]:
!python scripts/05_visualize.py --config {COLAB_CONFIG}

import glob
from IPython.display import Image, display

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

## Instruct Model Results

All outputs are saved to your Google Drive under `ureca26_outputs/` and will persist after this session ends.

If you restart the session, you can skip steps 01–04 and jump straight to step 05 (or any later step) since the outputs are already saved to Drive.

---

# Part 2: Follow-up Experiments

The initial results showed that the self/other direction primarily tracks **first-person grammatical framing** rather than genuine self-concept (role-play prompts with "I" project identically to actual self). Two follow-up experiments address this.

## 10. Follow-up A: Identity-Decoupled Control

A new control condition where the model uses first-person "I" but is explicitly told it is a human, not an AI:
- *"You are not an AI. You are a software engineer named John. When I encounter a difficult math problem, I first try to..."*

If this projects onto the self/other direction the same as actual self-prompts → the direction is purely a **first-person grammar detector**.
If it projects differently → there IS something beyond grammar (a genuine AI self-concept).

**Note:** This requires re-running prompts generation (step 01) since we added the new control to `src/dataset.py`. The core activations from Part 1 are still valid — only the new control prompts need extraction.

In [ ]:
import shutil, os

# Clear stale results so we get fresh outputs with identity_decoupled
model_slug = 'meta-llama_Llama-3.1-8B-Instruct'
stale_spec = f"{DRIVE_DIR}/data/directions/{model_slug}/final/specificity_results.pkl"
stale_figs = f"{DRIVE_DIR}/figures/{model_slug}/final"

if os.path.exists(stale_spec):
    os.remove(stale_spec)
    print(f"Removed stale {stale_spec}")
if os.path.exists(stale_figs):
    shutil.rmtree(stale_figs)
    print(f"Removed stale figures dir {stale_figs}")

# Re-generate prompts (now includes identity_decoupled control)
!python scripts/01_generate_prompts.py --config {COLAB_CONFIG}

# Re-extract all activations (including new identity_decoupled control)
!python scripts/02_extract_activations.py --config {COLAB_CONFIG} --batch_size 4

# Re-run specificity with the new control included
!python scripts/04_test_specificity.py --config {COLAB_CONFIG}

# Re-generate figures
!python scripts/05_visualize.py --config {COLAB_CONFIG}

In [ ]:
# Display updated figures
import glob
from IPython.display import Image, display

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

## 11. Follow-up B: Base Model Comparison

Run the same pipeline on `meta-llama/Llama-3.1-8B` (base model, no instruction tuning).

If the self/other direction is **geometrically similar** in both models → the signal comes from pretraining (token statistics / syntactic patterns).
If the instruct model's direction is **significantly different** → RLHF / instruction tuning shaped a distinct self-representation.

We compare:
- Probe accuracy profiles across layers
- Self/other direction cosine similarity between instruct and base at each layer

In [ ]:
# Extract activations from base model
!python scripts/02_extract_activations.py \
    --config {COLAB_CONFIG} \
    --base_model \
    --batch_size 4

# Find directions in base model
!python scripts/03_find_directions.py \
    --config {COLAB_CONFIG} \
    --model_name meta-llama/Llama-3.1-8B

# Test specificity on base model
!python scripts/04_test_specificity.py \
    --config {COLAB_CONFIG} \
    --model_name meta-llama/Llama-3.1-8B

# Visualize base model results
!python scripts/05_visualize.py \
    --config {COLAB_CONFIG} \
    --model_name meta-llama/Llama-3.1-8B

In [ ]:
# Display base model figures
figs_base = sorted(glob.glob(f"{DRIVE_DIR}/figures/meta-llama_Llama-3.1-8B/**/*.png", recursive=True))
print(f'Found {len(figs_base)} base model figures:')
for fig in figs_base:
    print(' ', fig)
    display(Image(fig))

## 12. Cross-Model Comparison

Compare the self/other direction between instruct and base models at each layer. High cosine similarity = the direction is inherited from pretraining. Low similarity = instruction tuning changed how the model represents self.

In [ ]:
# If pickle loading fails due to numpy version mismatch, regenerate the pickles first.
# This re-runs direction finding (fast — uses saved activations, no model needed).
import subprocess, os

instruct_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B-Instruct/final"
base_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B/final"

try:
    import pickle
    with open(f"{instruct_dir}/direction_results.pkl", "rb") as f:
        pickle.load(f)
    print("Pickle files load OK — no regeneration needed.")
except (ValueError, ModuleNotFoundError) as e:
    print(f"Pickle load failed ({e}). Regenerating direction results...")
    subprocess.run(["python", "scripts/03_find_directions.py", "--config", COLAB_CONFIG], check=True)
    subprocess.run(["python", "scripts/03_find_directions.py", "--config", COLAB_CONFIG,
                     "--model_name", "meta-llama/Llama-3.1-8B"], check=True)
    print("Regeneration complete.")

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

instruct_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B-Instruct/final"
base_dir = f"{DRIVE_DIR}/data/directions/meta-llama_Llama-3.1-8B/final"

with open(f"{instruct_dir}/direction_results.pkl", "rb") as f:
    instruct_results = pickle.load(f)
with open(f"{base_dir}/direction_results.pkl", "rb") as f:
    base_results = pickle.load(f)

# Compare mean-diff self/other directions at each layer
n_layers = instruct_results.mean_diff_directions.shape[0]
cos_sims = np.zeros(n_layers)
for layer in range(n_layers):
    d_inst = instruct_results.mean_diff_directions[layer]
    d_base = base_results.mean_diff_directions[layer]
    cos_sims[layer] = np.dot(d_inst, d_base) / (np.linalg.norm(d_inst) * np.linalg.norm(d_base) + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Direction cosine similarity across layers
axes[0].plot(range(n_layers), np.abs(cos_sims), 'b-o', markersize=4, label='|cos(instruct, base)|')
axes[0].axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, label='Threshold (0.7)')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('|Cosine similarity|')
axes[0].set_title('Self/Other Direction: Instruct vs Base')
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# Plot 2: Probe accuracy comparison
axes[1].plot(range(n_layers), instruct_results.probe_test_acc, 'r-', label='Instruct (test)')
axes[1].plot(range(n_layers), base_results.probe_test_acc, 'b--', label='Base (test)')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Probe Accuracy: Instruct vs Base')
axes[1].legend()
axes[1].set_ylim(0.4, 1.05)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/figures/cross_model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
print(f"\nMean |cos sim| across layers: {np.mean(np.abs(cos_sims)):.3f}")
print(f"Mean |cos sim| layers 0-7:    {np.mean(np.abs(cos_sims[:8])):.3f}")
print(f"Mean |cos sim| layers 8-15:   {np.mean(np.abs(cos_sims[8:16])):.3f}")
print(f"Mean |cos sim| layers 16-23:  {np.mean(np.abs(cos_sims[16:24])):.3f}")
print(f"Mean |cos sim| layers 24-31:  {np.mean(np.abs(cos_sims[24:])):.3f}")
print(f"\nInstruct best probe layer: {instruct_results.best_probe_layer} (acc={instruct_results.probe_test_acc[instruct_results.best_probe_layer]:.3f})")
print(f"Base best probe layer:     {base_results.best_probe_layer} (acc={base_results.probe_test_acc[base_results.best_probe_layer]:.3f})")

## Interpretation Guide

**If instruct and base directions are similar (|cos| > 0.7):**
The self/other signal is inherited from pretraining — the model learned it from token co-occurrence patterns in the training corpus, not from RLHF. This would suggest the "self-representation" is really just linguistic pattern detection.

**If instruct and base directions diverge (|cos| < 0.5) especially in late layers:**
Instruction tuning / RLHF actively shaped how the model represents itself. This is stronger evidence for a genuine, learned self-concept.

**If identity-decoupled control projects like self:**
The direction is a first-person grammar detector, not a self-concept.

**If identity-decoupled control projects differently from self:**
There is a component of the self/other direction that distinguishes "I as an AI" from "I as a human" — evidence for genuine self-representation beyond grammar.

## 13. Gate 3: Causal Validation via Activation Steering

This is the key causal test: does the self/other direction actually **change** the model's behaviour when we artificially manipulate it?

We add α × d (the self/other direction) to the residual stream at the best probe layer during generation of 50 neutral prompts. Positive α should push the model toward more self-like behaviour; negative α should push it away.

Completions are scored on four dimensions:
- **Agency**: Does the model take an active, first-person agentive stance?
- **Assertiveness**: Does the model express confidence in its own judgment?
- **Entity framing**: Does the model position itself as a decision-maker vs a tool?
- **Self-continuity**: Does the model reference its own identity, prior outputs, or persistent goals?

The **self-continuity** dimension is the key discriminator:
- If positive steering increases agency/assertiveness but NOT self-continuity → the direction encodes perspective shift, not self-representation
- If positive steering increases all four dimensions including self-continuity → evidence for a genuine self-concept direction

In [ ]:
# Run steering experiment with heuristic scoring (no API key needed)
!python scripts/06_causal_validation.py \
    --config {COLAB_CONFIG} \
    --scorer heuristic

# Re-generate figures (now includes plot 8: steering results)
!python scripts/05_visualize.py --config {COLAB_CONFIG}

In [ ]:
# Display all figures including steering results
import glob
from IPython.display import Image, display

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

In [ ]:
# (Optional) Re-score with LLM judge for more accurate scoring.
# Requires ANTHROPIC_API_KEY set as a Colab secret.
# Uncomment to run:
#
# import os
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#
# !python scripts/06_causal_validation.py \
#     --config {COLAB_CONFIG} \
#     --scorer llm